# ADNET — Baseline Reproduction & Real Predictions (Kaggle version)

**What this notebook does:** trains a small number of lightweight transfer-learning baselines (and a simplified ADNET-style dual-stream model) on the public Kaggle Alzheimer's MRI dataset, using 5-fold cross-validation, and saves real per-sample predictions for each fold.

**What this notebook does NOT do:** re-implement all 12 published comparator methods exactly as described in their original papers. That is a multi-week undertaking. This notebook instead trains a defensible, honestly-labeled subset (3 baselines + a simplified ADNET) so that at least some of the manuscript's statistical comparisons rest on real, reproducible predictions instead of literature-reported numbers. Use the results to correctly label which rows of Table 7/9 are "reproduced in-house" (these) vs. "as originally published" (everything else) — per the response letter's Reviewer 1, Point 4.

**Setup steps on Kaggle (do these before running):**
1. Create a new notebook at https://www.kaggle.com/code → **New Notebook**
2. Upload this file via **File → Import Notebook**
3. On the right-hand panel, click **Add Input** → search for **"Alzheimer's Dataset (4 class of Images)"** (by tourist55) → click **Add**. No download step needed — it mounts directly into the notebook.
4. In the right-hand **Settings** panel, set **Accelerator** to **GPU T4 x2** (or P100 if offered) — this is required, training will be extremely slow on CPU.
5. Also in Settings, make sure **Internet** is turned **On** (needed to download pretrained ImageNet weights the first time each model is built).
6. Click **Run All**.

**Time and Kaggle's quota:** each model trains in roughly 15–30 minutes per fold on a Kaggle T4 GPU (less if you reduce epochs). With 5 folds × 4 models, budget 3–6 hours total. Kaggle gives free accounts **~30 GPU-hours per week** and each individual session can run up to **12 hours** — comfortably enough for this in one or two sessions. If you run out of weekly quota partway through, the checkpointing below lets you resume next week without re-training what's already done — but note that Kaggle notebook sessions do NOT persist `/kaggle/working` between separate interactive sessions unless you explicitly save it as a Dataset output (see the note at Section 5).

## 1. Setup

In [ ]:
!pip install timm --quiet
import torch, timm, os, glob, random, json
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type != 'cuda':
    print('WARNING: no GPU detected. Go to the Settings panel (right side) -> Accelerator -> GPU T4 x2, then re-run.')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Locate the dataset under /kaggle/input, and remove exact-duplicate copies

**Note:** this dataset mirror is known to contain an exact duplicate copy of every image (confirmed separately via the duplicate-check notebook: 12,800 raw files -> 6,400 truly unique images). The cell below removes those duplicates *before* creating the cross-validation folds, so a duplicate of a test-fold image can never leak into a training fold.

In [ ]:
print('Contents of /kaggle/input:')
for entry in os.listdir('/kaggle/input'):
    print(' -', entry)

if not os.listdir('/kaggle/input'):
    raise RuntimeError(
        "No dataset found under /kaggle/input. Click 'Add Input' in the right-hand "
        "panel and add the 'Alzheimer's Dataset (4 class of Images)' dataset, then "
        "re-run this cell."
    )

dataset_path = os.path.join('/kaggle/input', os.listdir('/kaggle/input')[0])
print('\nUsing dataset path:', dataset_path)

class_dirs = []
for root, dirs, files in os.walk(dataset_path):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        class_dirs.append((root, imgs))

CLASS_NAMES = sorted(set(os.path.basename(d) for d, _ in class_dirs))
print('Classes found:', CLASS_NAMES)
assert len(CLASS_NAMES) == 4, f'Expected 4 classes, found {len(CLASS_NAMES)}: {CLASS_NAMES}'

raw_paths, raw_labels = [], []
for d, imgs in class_dirs:
    cls = os.path.basename(d)
    cls_idx = CLASS_NAMES.index(cls)
    for f in imgs:
        raw_paths.append(os.path.join(d, f))
        raw_labels.append(cls_idx)

print(f'Total raw images found: {len(raw_paths)}')

# --- Deduplicate before splitting ---
# We already found (via the separate duplicate-check notebook) that this
# dataset mirror contains exact duplicate copies of every image (12,800 raw
# -> 6,400 unique). Training/evaluating without removing these first would
# let a duplicate of a test-fold image sit in the training fold, silently
# inflating accuracy. This step removes that risk before any splitting happens.
print('\nDeduplicating exact-duplicate images before splitting (this takes a minute or two)...')
!pip install imagehash --quiet
import imagehash
from PIL import Image as PILImage

seen_hashes = {}
all_paths_list, all_labels_list = [], []
for path, label in zip(raw_paths, raw_labels):
    try:
        with PILImage.open(path) as img:
            h = str(imagehash.phash(img, hash_size=16))
    except Exception:
        continue
    if h not in seen_hashes:
        seen_hashes[h] = path
        all_paths_list.append(path)
        all_labels_list.append(label)

all_paths = np.array(all_paths_list)
all_labels = np.array(all_labels_list)
print(f'Unique images after deduplication: {len(all_paths)}  (removed {len(raw_paths) - len(all_paths)} exact duplicates)')
print('(For reference, the manuscript\'s Table 2 states 6,400 total images.)')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c}: {(all_labels == i).sum()}')

## 3. Dataset class and augmentation

In [ ]:
from torchvision import transforms

IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class MRIDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')
        img = self.transform(img)
        return img, self.labels[idx], self.paths[idx]

## 4. Models

Three published-style baselines (simplified to train quickly) plus a simplified ADNET-style dual-stream model.

In [ ]:
def make_baseline(name, num_classes=4):
    """Simple transfer-learning baselines: a frozen pretrained backbone + new classifier head."""
    if name == 'resnet50':
        m = timm.create_model('resnet50', pretrained=True, num_classes=num_classes)
    elif name == 'densenet121':
        m = timm.create_model('densenet121', pretrained=True, num_classes=num_classes)
    elif name == 'efficientnet_b0':
        m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
    else:
        raise ValueError(name)
    return m


class SimplifiedADNET(nn.Module):
    """
    A simplified dual-stream model in the spirit of the manuscript's ADNET
    (Section 4): an EfficientNet-B3 stream and a Swin-Tiny stream, concatenated
    and classified. This is NOT a full reproduction of every module described
    in the paper (no explicit CSPA/HAF here) -- it is a reasonable, quick-to-
    train proxy for getting real predictions to compare against baselines.
    For a full reproduction, use src/adnet_model.py from the Zenodo package
    instead (slower to train, more faithful to the manuscript's equations).
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.stream_a = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        self.stream_b = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        in_dim = self.stream_a.num_features + self.stream_b.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        fa = self.stream_a(x)
        fb = self.stream_b(x)
        return self.classifier(torch.cat([fa, fb], dim=1))


MODEL_BUILDERS = {
    'ResNet50 (in-house)': lambda: make_baseline('resnet50'),
    'DenseNet121 (in-house)': lambda: make_baseline('densenet121'),
    'EfficientNet-B0 (in-house)': lambda: make_baseline('efficientnet_b0'),
    'ADNET-simplified (in-house)': lambda: SimplifiedADNET(),
}
print('Models to train:', list(MODEL_BUILDERS.keys()))

## 5. Training loop (5-fold cross-validation)

Reduce `EPOCHS` if you're short on time/GPU quota — even 3–5 epochs of fine-tuning a pretrained backbone gives a real, honest result, just not a fully converged one.

**Important Kaggle-specific note:** `/kaggle/working` is only guaranteed to persist for the current interactive session. If your session disconnects or you close the tab, in-progress checkpoints may be lost unless you click **Save Version** (top right) before stopping, which snapshots `/kaggle/working` as a permanent Kaggle Dataset/Output you can reopen and continue from. If you expect to need more than one sitting: click **Save Version → Save & Run All (Commit)** periodically, or after each fold completes, so partial progress is never lost.

In [ ]:
EPOCHS = 5          # increase for better-trained (but slower) models
BATCH_SIZE = 32
N_FOLDS = 5
CKPT_DIR = '/kaggle/working/checkpoints'
PRED_DIR = '/kaggle/working/predictions'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def train_one_fold(model_name, fold_idx, train_idx, test_idx):
    ckpt_path = os.path.join(CKPT_DIR, f'{model_name.replace(" ", "_")}_fold{fold_idx}.pt')
    pred_path = os.path.join(PRED_DIR, f'{model_name.replace(" ", "_")}_fold{fold_idx}.json')

    if os.path.exists(pred_path):
        print(f'  [skip] {model_name} fold {fold_idx} already done')
        return json.load(open(pred_path))

    train_ds = MRIDataset(all_paths[train_idx], all_labels[train_idx], train_tf)
    test_ds = MRIDataset(all_paths[test_idx], all_labels[test_idx], eval_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = MODEL_BUILDERS[model_name]().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f'  Resumed {model_name} fold {fold_idx} from checkpoint')
    else:
        model.train()
        for epoch in range(EPOCHS):
            total_loss = 0.0
            for imgs, labels, _ in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                opt.zero_grad()
                out = model(imgs)
                loss = crit(out, labels)
                loss.backward()
                opt.step()
                total_loss += loss.item()
            print(f'  {model_name} fold {fold_idx} epoch {epoch+1}/{EPOCHS} loss={total_loss/len(train_loader):.4f}')
        torch.save(model.state_dict(), ckpt_path)

    model.eval()
    all_preds, all_true, all_paths_out = [], [], []
    with torch.no_grad():
        for imgs, labels, paths in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds = out.argmax(dim=1).cpu().numpy().tolist()
            all_preds.extend(preds)
            all_true.extend(labels.numpy().tolist())
            all_paths_out.extend(list(paths))

    result = {'model': model_name, 'fold': fold_idx, 'preds': all_preds, 'true': all_true, 'paths': all_paths_out}
    json.dump(result, open(pred_path, 'w'))
    del model
    torch.cuda.empty_cache()
    return result


all_results = {name: [] for name in MODEL_BUILDERS}
for model_name in MODEL_BUILDERS:
    print(f'\n=== {model_name} ===')
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_paths, all_labels)):
        result = train_one_fold(model_name, fold_idx, train_idx, test_idx)
        all_results[model_name].append(result)

print('\nAll folds complete for all models.')

## 6. Per-model accuracy summary

In [ ]:
print(f'{"Model":<30} {"Mean Acc":>10} {"Std":>8} {"Mean F1 (macro)":>18}')
summary = {}
for model_name, folds in all_results.items():
    accs, f1s = [], []
    for r in folds:
        accs.append(accuracy_score(r['true'], r['preds']))
        f1s.append(f1_score(r['true'], r['preds'], average='macro'))
    summary[model_name] = {'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)), 'f1_mean': float(np.mean(f1s))}
    print(f'{model_name:<30} {np.mean(accs)*100:>9.2f}% {np.std(accs)*100:>7.2f}% {np.mean(f1s)*100:>17.2f}%')

json.dump(summary, open('/kaggle/working/summary.json', 'w'), indent=2)

## 7. McNemar's test: ADNET-simplified vs. each in-house baseline

This is real, paired, per-sample statistical testing — exactly what Reviewer 1's Point 4 asked for. Pools predictions across all 5 folds per model (matched by image path) before testing.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def pooled_correctness(model_name):
    """Return {image_path: is_correct} pooled across all folds."""
    out = {}
    for r in all_results[model_name]:
        for path, pred, true in zip(r['paths'], r['preds'], r['true']):
            out[path] = int(pred == true)
    return out

adnet_correct = pooled_correctness('ADNET-simplified (in-house)')

print(f'{"Comparison":<45} {"chi2":>8} {"p-value":>12} {"n paired":>10}')
mcnemar_results = {}
for model_name in MODEL_BUILDERS:
    if model_name == 'ADNET-simplified (in-house)':
        continue
    baseline_correct = pooled_correctness(model_name)
    common_paths = sorted(set(adnet_correct) & set(baseline_correct))

    # 2x2 contingency table: [both correct, adnet correct only; baseline correct only, both wrong]
    both_correct = sum(1 for p in common_paths if adnet_correct[p] and baseline_correct[p])
    adnet_only = sum(1 for p in common_paths if adnet_correct[p] and not baseline_correct[p])
    baseline_only = sum(1 for p in common_paths if not adnet_correct[p] and baseline_correct[p])
    both_wrong = sum(1 for p in common_paths if not adnet_correct[p] and not baseline_correct[p])

    table = [[both_correct, adnet_only], [baseline_only, both_wrong]]
    result = mcnemar(table, exact=(adnet_only + baseline_only) < 25, correction=True)
    mcnemar_results[model_name] = {'chi2': float(result.statistic), 'pvalue': float(result.pvalue), 'n': len(common_paths)}
    print(f'ADNET-simplified vs. {model_name:<25} {result.statistic:>8.2f} {result.pvalue:>12.5f} {len(common_paths):>10}')

json.dump(mcnemar_results, open('/kaggle/working/mcnemar_results.json', 'w'), indent=2)
print('\nCOPY THE TABLE ABOVE (from "Comparison" header down) and send it back to replace')
print('Table 9\'s [PROVISIONAL] rows for these specific models.')
print('Note: Bonferroni-corrected alpha for 3 comparisons = 0.05/3 = 0.0167.')

## 8. Moderate Demented pooled confidence interval (Item 7 in the playbook)

Pools the Moderate Demented class predictions from ADNET-simplified across all 5 folds for a tighter, more defensible confidence interval than the single-fold number in the current manuscript.

In [ ]:
from scipy import stats as scipy_stats

MOD_DEMENTED_IDX = CLASS_NAMES.index([c for c in CLASS_NAMES if 'oderate' in c][0])

n_correct, n_total = 0, 0
for r in all_results['ADNET-simplified (in-house)']:
    for pred, true in zip(r['preds'], r['true']):
        if true == MOD_DEMENTED_IDX:
            n_total += 1
            n_correct += int(pred == true)

acc = n_correct / n_total if n_total else float('nan')
if n_total:
    # Standard Clopper-Pearson exact binomial CI, handling the k=0 and k=n
    # edge cases explicitly (the general beta-distribution formula is
    # undefined at those boundaries since it needs both shape parameters > 0).
    alpha = 0.05
    lo = 0.0 if n_correct == 0 else scipy_stats.beta.ppf(alpha / 2, n_correct, n_total - n_correct + 1)
    hi = 1.0 if n_correct == n_total else scipy_stats.beta.ppf(1 - alpha / 2, n_correct + 1, n_total - n_correct)
    print(f'Moderate Demented, pooled across 5 folds: {n_correct}/{n_total} = {acc*100:.1f}%')
    print(f'95% exact (Clopper-Pearson) CI: [{lo*100:.1f}%, {hi*100:.1f}%]')
    print('\nCOPY THIS LINE and send it back to replace the single-fold n=9-10 CI in the Abstract and Table 6.')
else:
    print('No Moderate Demented samples found -- check CLASS_NAMES matching above.')

---
### What to send back
1. The accuracy/F1 summary table from Section 6
2. The McNemar comparison table from Section 7
3. The pooled Moderate Demented CI line from Section 8

All three will be integrated directly into the manuscript, replacing the [PROVISIONAL] flags in Table 9 (for these specific models only — the other baselines not trained here remain literature comparisons) and the single-fold CI in the Abstract/Table 6.

**Retrieving files from Kaggle:** everything under `/kaggle/working` (checkpoints, prediction JSONs, summary.json, mcnemar_results.json) appears in the notebook's **Output** tab once you click **Save Version** (top right) and the run completes. From there each file has its own download button, and the whole output folder can also be downloaded as a zip.